# S16 · The best portfolios — the efficient frontier

We ask the computer: for each level of return we might want, what is the calmest
possible mix of stocks? Each answer is one best portfolio. Joining all of them
traces a curve called the **efficient frontier**: the most return you can get at
each level of risk. We then mark the two portfolios everyone talks about, the
calmest one and the best reward-per-risk one.

**New here? Read this once.**

- New to Python? You can still run the whole notebook with the play button and
  read the plain-English note above each cell.
- New to the idea of "least risk for a target return"? The primer
  `primers/diversification_and_covariance.md` sets up risk and covariance in ten
  minutes.
- Already confident? Look for the cells marked **Stretch (optional)**.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses cvxpy (to solve the optimisation) and yfinance (to fetch
# prices). Colab ships neither reliably, so we install them there.
import sys
if "google.colab" in sys.modules:
    !pip install -q cvxpy yfinance
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — get our tools ready

The same tools as notebook 1, plus **CVXPY**. CVXPY is a library where you describe
an optimisation problem in plain, maths-like lines, and it finds the answer for
you. You never write the search algorithm yourself.

In [ ]:
import numpy as np                 # arrays and maths
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # charts
import cvxpy as cp                 # states and solves optimisation problems for us

np.random.seed(0)
print("Tools ready, including CVXPY.")

## Step 2 — get prices (same helper as before)

We reuse the data helper from notebook 1: download real prices, or make up
realistic ones offline. Nothing new here, so read it quickly.

In [ ]:
# We try to download REAL daily prices from Yahoo Finance.
# If there is no internet, we instead build a synthetic - made up but realistic -
# set of prices, so this notebook ALWAYS runs. We print which one happened.

# The stocks we want. The '.NS' suffix means the National Stock Exchange of India.
stock_tickers = ["TCS.NS", "INFY.NS", "RELIANCE.NS", "ITC.NS", "HDFCBANK.NS"]

used_live_data = False   # we will flip this to True if the download works

try:
    import yfinance as yf

    # Download about three years of daily prices for all the stocks at once.
    raw = yf.download(stock_tickers, period="3y", auto_adjust=True, progress=False)

    # yfinance gives us several price types (Open, High, Low, Close...).
    # We keep the 'Close' price - the price at the end of each trading day.
    prices = raw["Close"]

    # Drop any days where a price is missing, and keep the columns in our order.
    prices = prices.dropna()
    prices = prices[stock_tickers]

    # If we somehow got an empty table, treat it as a failure.
    if len(prices) < 50:
        raise ValueError("not enough price data came back")

    used_live_data = True
    print("Downloaded LIVE price data from Yahoo Finance.")

except Exception as problem:
    # ---- Offline fallback: build synthetic prices with NumPy. -------------
    print("Could not download live data, so we are using SYNTHETIC data instead.")
    print("(Reason:", problem, ")")

    number_of_stocks = len(stock_tickers)
    number_of_days = 750

    # Each stock has its own average daily return and its own wobble (risk).
    yearly_return = np.random.uniform(0.06, 0.20, size=number_of_stocks)
    yearly_risk = np.random.uniform(0.18, 0.40, size=number_of_stocks)
    daily_average_return = yearly_return / 252.0      # ~252 trading days a year
    daily_risk = yearly_risk / np.sqrt(252.0)

    # To make the stocks move TOGETHER a little (as real stocks do), we mix a few
    # shared 'market mood' signals into each stock's daily return.
    number_of_shared_factors = 3
    loadings = np.random.uniform(0.2, 1.0, size=(number_of_stocks, number_of_shared_factors))
    shared_moods = np.random.normal(0, 1, size=(number_of_days, number_of_shared_factors))
    own_wiggle = np.random.normal(0, 1, size=(number_of_days, number_of_stocks))

    daily_returns_matrix = shared_moods @ loadings.T + own_wiggle

    # Scale each column so it has the daily average return and risk we chose.
    for stock_number in range(number_of_stocks):
        column = daily_returns_matrix[:, stock_number]
        column = (column - column.mean()) / column.std()      # standardise
        column = column * daily_risk[stock_number] + daily_average_return[stock_number]
        daily_returns_matrix[:, stock_number] = column

    price_paths = 100 * np.cumprod(1 + daily_returns_matrix, axis=0)
    business_days = pd.bdate_range(end=pd.Timestamp.today(), periods=number_of_days)
    prices = pd.DataFrame(price_paths, index=business_days, columns=stock_tickers)

print("\nPrice table shape (rows = days, columns = stocks):", prices.shape)
prices.head()

## Step 3 — expected returns μ and covariance Σ

Exactly as in notebook 1: average the daily returns for μ, take the covariance for
Σ, and annualise both. We turn them into plain NumPy arrays because CVXPY works
with arrays.

In [ ]:
trading_days_per_year = 252

daily_returns = prices.pct_change().dropna()

# mu = expected yearly return of each stock (a 1D array).
mu = (daily_returns.mean() * trading_days_per_year).values

# Sigma = annualised covariance matrix (a 2D array).
Sigma = (daily_returns.cov() * trading_days_per_year).values

number_of_stocks = len(mu)
stock_names = list(prices.columns)

print("mu (expected yearly returns, %):", (mu * 100).round(2))
print("Sigma shape:", Sigma.shape)

## Step 4 — a quick health check on Σ

Before we optimise, one sanity check. Whatever mix we choose, its risk must never
come out negative, because a negative risk would be nonsense. There is a quick test
for this: a set of numbers called the *eigenvalues* of Σ should all be zero or
positive. We just check the smallest one. If it passes, the optimisation is a clean
single bowl with one best answer, and the solver will find it reliably.

In [ ]:
# The eigenvalues of Sigma. For a healthy covariance matrix they are all >= 0.
# (Tiny negatives like -1e-18 are just rounding noise and are fine.)
smallest_eigenvalue = np.linalg.eigvalsh(Sigma).min()

print("Smallest eigenvalue of Sigma:", smallest_eigenvalue)

if smallest_eigenvalue > -1e-8:
    print("Healthy: every mix will have a sensible, non-negative risk. Good to go.")
else:
    print("Warning: a clearly negative eigenvalue means Sigma is broken.")

### Stretch (optional) — what that check really is

For the mathematically curious: a covariance matrix should be **positive
semi-definite**, which means `wᵀΣw ≥ 0` for every weight vector `w`. That is just
"every portfolio has non-negative variance". The clean test is that all the
eigenvalues of Σ are ≥ 0, which is what we checked above. Positive semi-definite
also means the risk surface is a convex bowl, so the minimisation has a single
global best answer with no traps. None of this is needed to finish the lab; it is
the reason the lab works so smoothly.

## Step 5 — one optimisation: least risk for a target return

This is the heart of the session. We ask CVXPY, in three plain lines, to:

> *make the risk `wᵀΣw` as small as possible*,
> *while* the weights sum to 1, no weight is negative (no borrowing to short-sell),
> *and* the expected return reaches at least some target we choose.

Notice how each line of code reads almost like the sentence above.

In [ ]:
# 'w' is the unknown we are solving for: the weight on each stock.
w = cp.Variable(number_of_stocks)

# We will demand a target return; start with the average of all stock returns.
target_return = mu.mean()

# The portfolio's risk (its variance). cp.quad_form is how CVXPY writes w @ Sigma @ w.
portfolio_variance = cp.quad_form(w, Sigma)

# The rules the weights must obey.
constraints = [
    cp.sum(w) == 1,           # all the money is invested (weights sum to 1)
    w >= 0,                   # no short selling (every weight is zero or positive)
    mu @ w >= target_return,  # the mix must reach at least the target return
]

# Build the problem: make the risk as small as possible, obeying the rules.
problem = cp.Problem(cp.Minimize(portfolio_variance), constraints)

# Solve it. CVXPY picks a solver and finds the best weights.
problem.solve()

print("Solver status:", problem.status)
print("Best weights for this target (%):", (w.value * 100).round(2))
print("Resulting risk (volatility, %)  :", round(np.sqrt(w.value @ Sigma @ w.value) * 100, 2))

## Step 6 — wrap that in a small function

To draw the whole frontier we must solve the problem above many times, once for
each target return. So we put it in a function we can call in a loop. Given a target
return, it hands back the best weights.

In [ ]:
def best_weights_for_target(target):
    # Same problem as Step 5, but the target return is now an input.
    w = cp.Variable(number_of_stocks)
    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        mu @ w >= target,
    ]
    problem = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), constraints)
    problem.solve()

    # If the solver could not find an answer, return None.
    if w.value is None:
        return None
    return w.value

# Quick test: ask for the average return and print the weights.
test_weights = best_weights_for_target(mu.mean())
print("Test weights (%):", (test_weights * 100).round(2))

## Step 7 — trace the efficient frontier

We sweep the target return from the lowest stock return up to the highest. For each
target we find the least-risk mix and record its risk and return. Joining these
points gives the **efficient frontier**: the best return available at each level of
risk.

In [ ]:
# A list of target returns from the smallest to the largest stock return.
target_returns = np.linspace(mu.min(), mu.max(), 40)

frontier_risk = []      # each best portfolio's risk goes here
frontier_return = []    # ... and its return here

for target in target_returns:
    weights = best_weights_for_target(target)
    if weights is None:
        continue   # skip any target the solver could not reach

    # Compute the risk and return of this best portfolio.
    this_return = weights @ mu
    this_risk = np.sqrt(weights @ Sigma @ weights)

    frontier_return.append(this_return)
    frontier_risk.append(this_risk)

# Turn the lists into arrays for easy plotting.
frontier_risk = np.array(frontier_risk)
frontier_return = np.array(frontier_return)

print("We computed", len(frontier_risk), "portfolios along the frontier.")

## Step 8 — scatter many random portfolios

To prove the frontier really is the *best* you can do, we also throw 3000 random
mixes at the problem and plot them. Every random mix should land **below** the
frontier: none of them can beat it.

In [ ]:
number_of_random_portfolios = 3000

random_risk = []
random_return = []

for one_portfolio in range(number_of_random_portfolios):
    # Random positive weights that sum to 1 (a random recipe).
    random_weights = np.random.random(number_of_stocks)
    random_weights = random_weights / random_weights.sum()

    random_return.append(random_weights @ mu)
    random_risk.append(np.sqrt(random_weights @ Sigma @ random_weights))

random_risk = np.array(random_risk)
random_return = np.array(random_return)

print("Generated", number_of_random_portfolios, "random portfolios to compare against.")

## Step 9 — find the minimum-variance portfolio

The **minimum-variance** portfolio is the single calmest mix of all: the least risk,
ignoring return entirely. We find it with one more CVXPY problem, minimising the
risk with no return target at all.

In [ ]:
w = cp.Variable(number_of_stocks)
constraints = [cp.sum(w) == 1, w >= 0]      # just: invest everything, no shorting
problem = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), constraints)
problem.solve()

min_var_weights = w.value
min_var_return = min_var_weights @ mu
min_var_risk = np.sqrt(min_var_weights @ Sigma @ min_var_weights)

print("Minimum-variance portfolio:")
print("  return:", round(min_var_return * 100, 2), "%")
print("  risk  :", round(min_var_risk * 100, 2), "%")

## Step 10 — find the maximum-Sharpe portfolio

The **Sharpe ratio** measures reward per unit of risk: how much return you earn
above a safe rate, divided by the risk you took. `(return - safe rate) / risk`. The
portfolio with the biggest Sharpe ratio gives the most reward for the risk. We look
along our frontier and pick the point with the highest Sharpe ratio.

In [ ]:
risk_free_rate = 0.06    # a safe yearly return, e.g. a government bond (6%)

# Sharpe ratio of every portfolio on the frontier.
frontier_sharpe = (frontier_return - risk_free_rate) / frontier_risk

# The position of the biggest Sharpe ratio.
best_position = np.argmax(frontier_sharpe)

max_sharpe_risk = frontier_risk[best_position]
max_sharpe_return = frontier_return[best_position]
max_sharpe_value = frontier_sharpe[best_position]

print("Maximum-Sharpe portfolio on the frontier:")
print("  return:", round(max_sharpe_return * 100, 2), "%")
print("  risk  :", round(max_sharpe_risk * 100, 2), "%")
print("  Sharpe ratio:", round(max_sharpe_value, 3))

## Step 11 — put it all together in one picture

Now we plot everything: the cloud of random mixes, the efficient frontier curve
riding along their top edge, and the two special portfolios marked. This is the
classic picture of portfolio theory.

In [ ]:
plt.figure(figsize=(9, 6))

# The random portfolios, coloured by their Sharpe ratio.
random_sharpe = (random_return - risk_free_rate) / random_risk
scatter = plt.scatter(random_risk * 100, random_return * 100,
                      c=random_sharpe, cmap="viridis", s=8, alpha=0.5)
plt.colorbar(scatter, label="Sharpe ratio")

# The efficient frontier curve.
plt.plot(frontier_risk * 100, frontier_return * 100,
         color="#C0392B", linewidth=3, label="efficient frontier")

# The minimum-variance portfolio (a green circle).
plt.scatter(min_var_risk * 100, min_var_return * 100,
            color="#27AE60", s=200, marker="o",
            edgecolor="black", zorder=5, label="minimum variance")

# The maximum-Sharpe portfolio (a yellow star).
plt.scatter(max_sharpe_risk * 100, max_sharpe_return * 100,
            color="#F1C40F", s=300, marker="*",
            edgecolor="black", zorder=5, label="maximum Sharpe")

plt.xlabel("risk (volatility, %)")
plt.ylabel("expected return (%)")
plt.title("The efficient frontier")
plt.legend()
plt.show()

### Stretch (optional) — the same random cloud, the fast way

Skip this if you are new to code. The loop in Step 8 is easy to read, but experienced
NumPy users would generate all 3000 random portfolios at once, with no Python loop.
`np.random.dirichlet` draws weight rows that already sum to 1, and `np.einsum`
computes every mix's variance in one call. Same result, far faster.

In [ ]:
# Draw 3000 weight rows that each sum to 1, in one call.
random_weight_rows = np.random.dirichlet(np.ones(number_of_stocks), size=3000)

# Return of every mix at once: each row of weights times mu.
fast_return = random_weight_rows @ mu

# Variance of every mix at once: row_i . Sigma . row_i for every row.
fast_variance = np.einsum("ij,jk,ik->i", random_weight_rows, Sigma, random_weight_rows)
fast_risk = np.sqrt(fast_variance)

print("Vectorised cloud:", fast_risk.shape[0], "portfolios computed with no Python loop.")
print("Best random Sharpe found:", round(((fast_return - risk_free_rate) / fast_risk).max(), 3))

## What you just did

You used CVXPY to make the risk `wᵀΣw` as small as possible, again and again, once
per target return, and traced the **efficient frontier**: the best return for each
level of risk. The random mixes all sat below it, confirming the frontier really is
the edge of what is possible. You also pinned down the calmest mix (minimum
variance) and the most reward-efficient one (maximum Sharpe). Next we run the whole
thing on a basket of ten NSE stocks.